# Question 3 — Corners

Do winning teams have a significantly higher average number of corners than losing teams?

**Objective 1.** Run the cells from top to bottom. This notebook loads the original CSV independently. Tables and graphs appear directly beneath their code cells.

## 1. Libraries and project paths

Use the project `.venv` kernel. Inline plotting keeps the figures inside this notebook.

In [1]:
# Display graphs directly below their notebook cells.
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

QUESTION = 'Do winning teams have a significantly higher average number of corners than losing teams?'
NUMBER = 3
# Locate the project whether this notebook starts in the root or its question folder.
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
BASE = ROOT / 'Question_3_Corners'
for directory in ['processed_data', 'figures', 'tables', 'results']:
    (BASE / directory).mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 35)
print(QUESTION)

Do winning teams have a significantly higher average number of corners than losing teams?


## 2. Load and inspect the raw CSV

The file has 28 columns and repeated stage/header rows. We inspect these before selecting match observations.

In [2]:
ROOT = Path.cwd()
while not (ROOT / 'data').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

path = ROOT / 'data' / 'raw' / 'World Cup - 2026 - Stats - Fixtures.csv'

# Keep every row initially because this CSV contains several header rows.
raw = pd.read_csv(path, header=None)

# Read the third row as column labels for inspection only;
# no match data are loaded here.
headers = pd.read_csv(path, header=2, nrows=0).columns.tolist()

print('Raw rows:', len(raw), '| Columns:', raw.shape[1])
display(raw.head())

Raw rows: 121 | Columns: 28


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27
0,GROUP STAGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,GOALS,NaN,NaN,NaN,CARDS,NaN,NaN,NaN,CORNERS,NaN,NaN,NaN,NaN,NaN,NaN,xG,NaN,SHOTS,NaN,SHOTS ON\r\nTARGET,NaN,FOULS,NaN
2,Date,Team 1,Result,NaN,Team 2,1H,NaN,2H,NaN,YEL,NaN,RED,NaN,1H C,NaN,2H C,NaN,MC,NaN,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11-06-26,Mexico,2,0,South Africa,1,0.0,1,0.0,1,2.0,1,2.0,2,1.0,1,0.0,3,1.0,4,1.46,0.07,16,3.0,4,2.0,12,11.0
4,12-06-26,South Korea,2,1,Czech Republic,0,0.0,2,1.0,1,0.0,0,0.0,3,3.0,1,2.0,4,5.0,9,2.3,0.83,15,7.0,6,4.0,9,16.0


In [3]:
# Combine column names, types and missing counts into one readable inspection table.
column_inspection = pd.DataFrame({
    'Position': range(raw.shape[1]),
    'Original pandas name (header=2)': headers,
    'Raw data type': raw.dtypes.astype(str).values,
    'Missing values': raw.isnull().sum().values
})
display(column_inspection)
print('Raw duplicate records (including repeated headers):', raw.duplicated().sum())

,Position,Original pandas name (header=2),Raw data type,Missing values
0,0,Date,object,6
1,1,Team 1,object,12
2,2,Result,object,12
3,3,Unnamed: 3,object,18
4,4,Team 2,object,12
5,5,1H,object,6
6,6,Unnamed: 6,float64,18
7,7,2H,object,12
8,8,Unnamed: 8,float64,18
9,9,YEL,object,6


Raw duplicate records (including repeated headers): 9


## 3. Extract match rows and identify stages

Stage labels are section headings. Carry them down to their matches and remove the repeated headers. No merge is required.

In [4]:
stages = ['GROUP STAGE', 'ROUND OF 32', 'ROUND OF 16',
          'QUARTER FINALS', 'SEMI FINALS', 'FINAL']

if raw.shape[1] != 28 or raw.iloc[2, 2] != 'Result' or raw.iloc[2, 17] != 'MC':
    raise ValueError('CSV layout changed: inspect its headers before analysing.')

# Keep recognised stage headings, then carry each heading down to their match rows.
raw['Stage'] = raw[0].where(raw[0].isin(stages)).ffill()

# A date-shaped first field identifies a match rather than a repeated header.
match_rows = raw[0].astype('str').str.fullmatch(r'\d{1,2}-\d{1,2}-\d{2}', na=False)
matches = raw.loc[match_rows].copy()

print('Match observations:', len(matches))
display(matches.head())
display(matches['Stage'].value_counts().rename('Matches').to_frame())

Match observations: 103


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,Stage
3,11-06-26,Mexico,2,0,South Africa,1,0.0,1,0.0,1,2.0,1,2.0,2,1.0,1,0.0,3,1.0,4,1.46,0.07,16,3.0,4,2.0,12,11.0,GROUP STAGE
4,12-06-26,South Korea,2,1,Czech Republic,0,0.0,2,1.0,1,0.0,0,0.0,3,3.0,1,2.0,4,5.0,9,2.3,0.83,15,7.0,6,4.0,9,16.0,GROUP STAGE
5,12-06-26,Canada,1,1,Bosnia & Herzegovina,0,1.0,1,0.0,2,3.0,0,0.0,9,1.0,0,3.0,9,4.0,13,1.25,0.98,13,8.0,4,3.0,10,20.0,GROUP STAGE
6,13-06-26,USA,4,1,Paraguay,3,0.0,1,1.0,1,5.0,0,0.0,2,0.0,1,1.0,3,1.0,4,1.42,0.54,16,9.0,6,1.0,13,17.0,GROUP STAGE
7,13-06-26,Qatar,1,1,Switzerland,0,1.0,1,0.0,2,1.0,0,0.0,3,4.0,0,6.0,3,10.0,13,0.6,3.20,6,26.0,3,7.0,12,11.0,GROUP STAGE


,Matches
Stage,
GROUP STAGE,72
ROUND OF 32,16
ROUND OF 16,8
QUARTER FINALS,4
SEMI FINALS,2
FINAL,1


## 4. Select the relevant columns

The aliases below are explicitly mapped to the inspected CSV positions. Scores use `Result` and its unnamed neighbour; corners use `MC` and its neighbour; yellow cards use `YEL` and its neighbour; target shots use the pair beneath `SHOTS ON\nTARGET`.

In [5]:
# Map verified zero-based CSV positions to readable names for this question.
mapping = {0: 'Date', 1: 'Team 1', 4: 'Team 2', 2: 'Score 1', 3: 'Score 2', 17: 'Corners 1', 18: 'Corners 2'}
data = matches[list(mapping) + ['Stage']].rename(columns=mapping).copy()

# Keep the original CSV record number so each selected match can be traced back.
data['Source record'] = data.index + 1

display(data.head(10))
display(pd.DataFrame({'Type': data.dtypes.astype(str), 'Missing': data.isnull().sum()}))

,Date,Team 1,Team 2,Score 1,Score 2,Corners 1,Corners 2,Stage,Source record
3,11-06-26,Mexico,South Africa,2,0,3,1.0,GROUP STAGE,4
4,12-06-26,South Korea,Czech Republic,2,1,4,5.0,GROUP STAGE,5
5,12-06-26,Canada,Bosnia & Herzegovina,1,1,9,4.0,GROUP STAGE,6
6,13-06-26,USA,Paraguay,4,1,3,1.0,GROUP STAGE,7
7,13-06-26,Qatar,Switzerland,1,1,3,10.0,GROUP STAGE,8
8,13-06-26,Brazil,Morocco,1,1,6,2.0,GROUP STAGE,9
9,14-06-26,Haiti,Scotland,0,1,4,3.0,GROUP STAGE,10
10,14-06-26,Australia,Turkey,2,0,5,8.0,GROUP STAGE,11
11,14-06-26,Germany,Curacao,7,1,8,1.0,GROUP STAGE,12
12,14-06-26,Netherlands,Japan,2,2,5,4.0,GROUP STAGE,13


,Type,Missing
Date,object,0
Team 1,object,0
Team 2,object,0
Score 1,object,0
Score 2,object,0
Corners 1,object,0
Corners 2,float64,0
Stage,object,0
Source record,int64,0


## 5. Clean required fields

Convert the score and corner columns to numeric values and remove rows missing the fields required for the analysis.

In [6]:
before = len(data)

# Only exact duplicate match records are removed. Conflicting identities stop the run.
exact_duplicates = matches.drop(columns='Stage').duplicated()
data = data.loc[~exact_duplicates].copy()

print('Exact duplicate matches removed: %d' % exact_duplicates.sum())

if data.duplicated(['Date', 'Team 1', 'Team 2']).any():
    raise ValueError('Conflicting duplicate match identities need review.')

# Parse day/month/year dates; errors="coerce" marks unparseable values as missing.
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True, errors='coerce')

# Remove extra spaces from team names and treat empty names as missing.
for column in ['Team 1', 'Team 2']:
    data[column] = data[column].str.strip().replace('', pd.NA)

numeric_columns = ['Score 1', 'Score 2', 'Corners 1', 'Corners 2']

for column in numeric_columns:
    values = data[column].astype('str')

    if column.startswith('Score'):
        # Only accept a numeric score with an optional leading/trailing p marker.
        # Retain the numeric match score, never add shootout kicks to goals.
        valid_score = values.str.fullmatch(r'(?:p\d+|\d+p?)', na=False)
        values = values.where(valid_score).str.replace('p', '', regex=False)

    data[column] = pd.to_numeric(values, errors='coerce')

# Flag a row if any required count is missing, infinite, negative or fractional.
invalid_numeric = (
    ~np.isfinite(data[numeric_columns]) |
    (data[numeric_columns] < 0) |
    (data[numeric_columns] % 1 != 0)
).any(axis=1)

# Check only the fields needed for this analysis, not every field in the original CSV.
missing_required = data[list(mapping.values()) + ['Stage']].isnull().any(axis=1)

print('Missing/unparseable required fields: %d' % missing_required.sum())
print('Rows with invalid, negative or noninteger counts: %d' % invalid_numeric.sum())

print('Rows excluded for invalid or missing required fields:')
display(data.loc[missing_required | invalid_numeric])

# The ~ symbol means NOT: keep rows that pass both cleaning checks.
data = data.loc[~(missing_required | invalid_numeric)].copy()

display(data.head(10))

Exact duplicate matches removed: 0
Missing/unparseable required fields: 0
Rows with invalid, negative or noninteger counts: 0
Rows excluded for invalid or missing required fields:


C:\Users\User\AppData\Local\Temp\ipykernel_7660\3129377267.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Date'] = pd.to_datetime(data['Date'], dayfirst=True, errors='coerce')


,Date,Team 1,Team 2,Score 1,Score 2,Corners 1,Corners 2,Stage,Source record


,Date,Team 1,Team 2,Score 1,Score 2,Corners 1,Corners 2,Stage,Source record
3,2026-06-11,Mexico,South Africa,2,0,3,1.0,GROUP STAGE,4
4,2026-06-12,South Korea,Czech Republic,2,1,4,5.0,GROUP STAGE,5
5,2026-06-12,Canada,Bosnia & Herzegovina,1,1,9,4.0,GROUP STAGE,6
6,2026-06-13,USA,Paraguay,4,1,3,1.0,GROUP STAGE,7
7,2026-06-13,Qatar,Switzerland,1,1,3,10.0,GROUP STAGE,8
8,2026-06-13,Brazil,Morocco,1,1,6,2.0,GROUP STAGE,9
9,2026-06-14,Haiti,Scotland,0,1,4,3.0,GROUP STAGE,10
10,2026-06-14,Australia,Turkey,2,0,5,8.0,GROUP STAGE,11
11,2026-06-14,Germany,Curacao,7,1,8,1.0,GROUP STAGE,12
12,2026-06-14,Netherlands,Japan,2,2,5,4.0,GROUP STAGE,13


## 6. Construct the analysis variable and eligible population

Keep both teams from each match together. Exclude equal recorded scores, including penalty-marked ties, as specified. This restricts the population to unequal-score matches; penalty shootouts may still have an eventual winner.

In [7]:
# Equal numeric scores are excluded here, including matches marked as penalty shootouts.
draws = data['Score 1'] == data['Score 2']
print('Equal-score matches excluded (including penalty-marked ties): %d' % draws.sum())

# This explicitly restricts the winner/loser population to unequal recorded scores.
data = data.loc[~draws].copy()

# This True/False condition identifies which side has the higher recorded score.
team1_won = data['Score 1'] > data['Score 2']

# np.where chooses Team 1 when the condition is True and Team 2 otherwise.
data['Winner'] = np.where(team1_won, data['Team 1'], data['Team 2'])
data['Loser'] = np.where(team1_won, data['Team 2'], data['Team 1'])

# Assign the corresponding corner totals to the winner and loser.
data['Winner Corners'] = np.where(team1_won, data['Corners 1'], data['Corners 2'])
data['Loser Corners'] = np.where(team1_won, data['Corners 2'], data['Corners 1'])

# A positive difference means the winner had more corners in that same match.
data['Corner Difference'] = data['Winner Corners'] - data['Loser Corners']

variables = ['Winner Corners', 'Loser Corners', 'Corner Difference']
population_description = 'All supplied unequal-score matches with valid corners and winner/loser information.'

print('Rows before cleaning: %d\nRows removed: %d\nRows after cleaning / eligible population: %d' %
      (before, before - len(data), len(data)))

print('Prepared data types:\n' + data.dtypes.to_string())

data.to_csv(BASE / 'processed_data' / ('question%d_population.csv' % NUMBER), index=False)

# Show every eligible observation, not just its dimensions.
with pd.option_context('display.max_rows', None):
    display(data)

Equal-score matches excluded (including penalty-marked ties): 24
Rows before cleaning: 103
Rows removed: 24
Rows after cleaning / eligible population: 79
Prepared data types:
Date                 datetime64[ns]
Team 1                       object
Team 2                       object
Score 1                       int64
Score 2                       int64
Corners 1                     int64
Corners 2                   float64
Stage                        object
Source record                 int64
Winner                       object
Loser                        object
Winner Corners              float64
Loser Corners               float64
Corner Difference           float64


,Date,Team 1,Team 2,Score 1,Score 2,Corners 1,Corners 2,Stage,Source record,Winner,Loser,Winner Corners,Loser Corners,Corner Difference
3,2026-06-11,Mexico,South Africa,2,0,3,1.0,GROUP STAGE,4,Mexico,South Africa,3.0,1.0,2.0
4,2026-06-12,South Korea,Czech Republic,2,1,4,5.0,GROUP STAGE,5,South Korea,Czech Republic,4.0,5.0,-1.0
6,2026-06-13,USA,Paraguay,4,1,3,1.0,GROUP STAGE,7,USA,Paraguay,3.0,1.0,2.0
9,2026-06-14,Haiti,Scotland,0,1,4,3.0,GROUP STAGE,10,Scotland,Haiti,3.0,4.0,-1.0
10,2026-06-14,Australia,Turkey,2,0,5,8.0,GROUP STAGE,11,Australia,Turkey,5.0,8.0,-3.0
11,2026-06-14,Germany,Curacao,7,1,8,1.0,GROUP STAGE,12,Germany,Curacao,8.0,1.0,7.0
13,2026-06-15,Ivory Coast,Ecuador,1,0,3,5.0,GROUP STAGE,14,Ivory Coast,Ecuador,3.0,5.0,-2.0
14,2026-06-15,Sweden,Tunisia,5,1,4,2.0,GROUP STAGE,15,Sweden,Tunisia,4.0,2.0,2.0
19,2026-06-16,France,Senegal,3,1,6,4.0,GROUP STAGE,20,France,Senegal,6.0,4.0,2.0
20,2026-06-16,Iraq,Norway,1,4,2,5.0,GROUP STAGE,21,Norway,Iraq,5.0,2.0,3.0
